In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto.')

for candidate in [
    PROJECT_ROOT / 'code',
    PROJECT_ROOT / 'code' / 'revenue',
    PROJECT_ROOT / 'code' / 'tmdb',
]:
    if candidate.exists() and str(candidate.resolve()) not in sys.path:
        sys.path.append(str(candidate.resolve()))

import pandas as pd

from experiment_utils import format_summary_display
from imbalance_experiment_utils import (
    ORACLE_BAND_ARTIFACT_DIR,
    ORACLE_LOCAL_MODEL_NAMES,
    TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
    build_summary_comparison,
    load_best_params_lookup_from_artifact_dir,
    load_results_from_artifact_dir,
    load_tmdb_extended_context,
    run_oracle_band_experiment,
    save_additional_table,
    save_artifact_tables,
    summarize_errors_by_band,
)


/home/gabriel/Faculdade/Matérias/ML/UFSJ_Aprendizado_Maquina_TP1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Modelos por Faixa de Arrecadação na Base TMDB Estendida com Oracle de Banda**

Este notebook testa uma estratégia diagnóstica para lidar com a assimetria de `revenue`: treinar regressões locais por faixa de arrecadação e, no teste, rotear cada filme usando **a faixa verdadeira**. Isso não representa um cenário de deploy, mas funciona como um limite superior para verificar se modelos especializados por banda têm potencial.

In [2]:
TARGET_NAME = 'Sem transformação'
OVERWRITE_ARTIFACTS = False
SHOW_PROGRESS = True
MIN_BAND_SAMPLES = 80

ARTIFACT_DIR = ORACLE_BAND_ARTIFACT_DIR
ERROR_ANALYSIS_DIR = ARTIFACT_DIR / 'error_analysis'
TRAINING_SIZES_PATH = ARTIFACT_DIR / 'training_band_sizes.csv'
SUMMARY_COMPARISON_PATH = ARTIFACT_DIR / 'comparison_with_global_models.csv'
BAND_METRICS_PATH = ERROR_ANALYSIS_DIR / '07_oracle_band_models_metricas_por_faixa.csv'


In [3]:
context = load_tmdb_extended_context()
df_movies = context['df_movies']
X = context['X']
y = context['y']
folds_df = context['folds_df']
revenue_bins = context['revenue_bins']

print(f'Base TMDB estendida: {df_movies.shape[0]} filmes | {X.shape[1]} features')
print(f'Folds carregados: {folds_df.shape[0]}')
display(df_movies[['id_tmdb', 'title', 'revenue']].head())


Base TMDB estendida: 6918 filmes | 284 features
Folds carregados: 10


,id_tmdb,title,revenue
0,552524,Lilo & Stitch,610800000
1,950387,A Minecraft Movie,947000000
2,1257960,सिकंदर,24727058
3,574475,Final Destination Bloodlines,229314062
4,1197306,A Working Man,98652557


In [4]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ERROR_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

if (
    not OVERWRITE_ARTIFACTS
    and (ARTIFACT_DIR / 'model_selection_results.csv').exists()
    and (ARTIFACT_DIR / 'model_selection_predictions.csv').exists()
    and (ARTIFACT_DIR / 'model_selection_summary.csv').exists()
    and TRAINING_SIZES_PATH.exists()
):
    results_df, predictions_df, summary_df = load_results_from_artifact_dir(ARTIFACT_DIR)
    training_sizes_df = pd.read_csv(TRAINING_SIZES_PATH)
else:
    best_params_lookup = load_best_params_lookup_from_artifact_dir(
        TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
        target_name=TARGET_NAME,
    )
    results_df, predictions_df, summary_df, training_sizes_df = run_oracle_band_experiment(
        df_movies=df_movies,
        X=X,
        y=y,
        folds_df=folds_df,
        revenue_bins=revenue_bins,
        best_params_lookup=best_params_lookup,
        model_names=ORACLE_LOCAL_MODEL_NAMES,
        target_name=TARGET_NAME,
        min_band_samples=MIN_BAND_SAMPLES,
        show_progress=SHOW_PROGRESS,
    )
    save_artifact_tables(
        ARTIFACT_DIR,
        results_df=results_df,
        predictions_df=predictions_df,
        summary_df=summary_df,
    )
    training_sizes_df.to_csv(TRAINING_SIZES_PATH, index=False)

global_results_df, global_predictions_df, global_summary_df = load_results_from_artifact_dir(
    TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
)
global_summary_df = global_summary_df.loc[
    (global_summary_df['target_version'] == TARGET_NAME)
    & (global_summary_df['model'].isin(ORACLE_LOCAL_MODEL_NAMES))
].copy()
summary_df = summary_df.loc[summary_df['target_version'] == TARGET_NAME].copy()

comparison_df = build_summary_comparison(
    global_summary_df,
    summary_df,
    reference_label='global',
    candidate_label='oracle',
)
save_additional_table(comparison_df, SUMMARY_COMPARISON_PATH)

best_oracle_model_name = summary_df.sort_values(['mean_rmse', 'mean_mae', 'model']).iloc[0]['model']
best_oracle_predictions_df = predictions_df.loc[predictions_df['model'] == best_oracle_model_name].copy()
band_metrics_df = summarize_errors_by_band(best_oracle_predictions_df, revenue_bins)
save_additional_table(band_metrics_df, BAND_METRICS_PATH)

display(format_summary_display(summary_df))
display(comparison_df[[
    'model',
    'mean_rmse_global',
    'mean_rmse_oracle',
    'delta_rmse',
    'mean_mae_global',
    'mean_mae_oracle',
    'delta_mae',
    'mean_r2_global',
    'mean_r2_oracle',
    'delta_r2',
]].sort_values('mean_rmse_oracle'))
display(band_metrics_df)


Oracle por faixa: 100%|██████████| 30/30 [05:56<00:00, 11.87s/fit]


MSE  \
Versão do alvo    Modelo                                               
Sem transformação Gradient Boosting Regressor  9.65e+15 (± 3.34e+15)   
                  XGBoost Regressor            9.76e+15 (± 3.56e+15)   
                  Random Forest Regressor      9.92e+15 (± 3.47e+15)   

                                                                RMSE  \
Versão do alvo    Modelo                                               
Sem transformação Gradient Boosting Regressor  96.85 mi (± 17.44 mi)   
                  XGBoost Regressor            97.44 mi (± 17.30 mi)   
                  Random Forest Regressor      98.19 mi (± 17.66 mi)   

                                                                MAE  \
Versão do alvo    Modelo                                              
Sem transformação Gradient Boosting Regressor  33.36 mi (± 2.56 mi)   
                  XGBoost Regressor            33.56 mi (± 2.41 mi)   
                  Random Forest Regressor      32.95 mi (± 2.63 mi)   

                                                            R²  
Versão do alvo    Modelo                                        
Sem transformação Gradient Boosting Regressor  0.720 (± 0.084)  
                  XGBoost Regressor            0.716 (± 0.084)  
                  Random Forest Regressor      0.707 (± 0.110)

,model,mean_rmse_global,mean_rmse_oracle,delta_rmse,mean_mae_global,mean_mae_oracle,delta_mae,mean_r2_global,mean_r2_oracle,delta_r2
1,Gradient Boosting Regressor,1.127063e+08,9.685453e+07,-1.585176e+07,5.569888e+07,3.335812e+07,-2.234076e+07,0.622988,0.719715,0.096727
0,XGBoost Regressor,1.116649e+08,9.744018e+07,-1.422472e+07,5.515751e+07,3.356488e+07,-2.159264e+07,0.630175,0.716468,0.086293
2,Random Forest Regressor,1.137802e+08,9.819428e+07,-1.558590e+07,5.522204e+07,3.294550e+07,-2.227654e+07,0.615321,0.707340,0.092018


,faixa_receita,quantidade_filmes,receita_minima,receita_maxima,mae_medio,residuo_medio,mediana_erro_absoluto,percentual_subestimados,percentual_superestimados,rmse
0,Muito baixa receita,1384,1,7086000,1.767151e+06,3.758812e+03,1.685947e+06,45.447977,54.552023,2.115918e+06
1,Baixa receita,1383,7096000,22441323,3.750624e+06,-5.112509e+03,3.543571e+06,47.650036,52.349964,4.452949e+06
2,Média receita,1384,22468044,52800000,7.673798e+06,6.153600e+04,7.427263e+06,48.265896,51.734104,9.021104e+06
3,Alta receita,1383,52900000,134038006,1.924105e+07,-1.705678e+05,1.805639e+07,46.854664,53.145336,2.315284e+07
4,Muito alta receita,1384,134100000,2923706026,1.343256e+08,-4.735344e+06,8.405004e+07,40.028902,59.971098,2.182152e+08


In [5]:
training_overview_df = (
    training_sizes_df.groupby(['model', 'faixa_receita'], as_index=False)
    .agg(
        media_amostras_treino=('n_train', 'mean'),
        min_amostras_treino=('n_train', 'min'),
        media_uso_fallback=('uses_fallback', 'mean'),
    )
)
display(training_overview_df)
print(f'Melhor modelo oracle: {best_oracle_model_name}')
print(f'CSV de comparação salvo em: {SUMMARY_COMPARISON_PATH}')
print(f'CSV por faixa salvo em: {BAND_METRICS_PATH}')


,model,faixa_receita,media_amostras_treino,min_amostras_treino,media_uso_fallback
0,Gradient Boosting Regressor,Alta receita,1244.7,1244,0.0
1,Gradient Boosting Regressor,Baixa receita,1244.7,1244,0.0
2,Gradient Boosting Regressor,Muito alta receita,1245.6,1245,0.0
3,Gradient Boosting Regressor,Muito baixa receita,1245.6,1245,0.0
4,Gradient Boosting Regressor,Média receita,1245.6,1245,0.0
5,Random Forest Regressor,Alta receita,1244.7,1244,0.0
6,Random Forest Regressor,Baixa receita,1244.7,1244,0.0
7,Random Forest Regressor,Muito alta receita,1245.6,1245,0.0
8,Random Forest Regressor,Muito baixa receita,1245.6,1245,0.0
9,Random Forest Regressor,Média receita,1245.6,1245,0.0


Melhor modelo oracle: Gradient Boosting Regressor
CSV de comparação salvo em: /home/gabriel/Faculdade/Matérias/ML/UFSJ_Aprendizado_Maquina_TP1/data/revenue_oracle_band_models_tmdb_extended/comparison_with_global_models.csv
CSV por faixa salvo em: /home/gabriel/Faculdade/Matérias/ML/UFSJ_Aprendizado_Maquina_TP1/data/revenue_oracle_band_models_tmdb_extended/error_analysis/07_oracle_band_models_metricas_por_faixa.csv
